# 01 — Cohort Characterization (Table 1)

Baseline demographics, comorbidity prevalence, and descriptive statistics for the three mutually exclusive new-user cohorts, after 1:5 PS matching.

**Study:** T2DM Persistence RWE — 30,000 synthetic patients, OMOP CDM v5.4  
**Design:** Active-comparator new-user cohort (Schneeweiss 2007, Lund 2015)  
**Investigator:** Zia Habibi

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import duckdb
from scipy.stats import kruskal

ROOT = Path('../../')
OUT_TABLES  = ROOT / 'outputs' / 'tables'
OUT_FIGURES = ROOT / 'outputs' / 'figures'
OUT_TABLES.mkdir(parents=True, exist_ok=True)
OUT_FIGURES.mkdir(parents=True, exist_ok=True)

cohort = pd.read_csv(OUT_TABLES / 'cohort_matched.csv')

COMORBIDITY_COLS = [
    'hypertension', 'obesity', 'ckd', 'heart_failure', 'hyperlipidemia',
    'nash', 'neuropathy', 'retinopathy', 'depression', 'atrial_fibrillation',
    'sleep_apnea', 'nafld', 'pvd', 'stroke', 'mi'
]
DRUG_CLASSES = ['metformin', 'glp1', 'sglt2']
drug_colors  = {'metformin': '#3498DB', 'glp1': '#E74C3C', 'sglt2': '#2ECC71'}

print(f'Matched cohort: {len(cohort):,} patients')
print(cohort['drug_class'].value_counts())

## OMOP CDM Cohort Definition SQL

The SQL below defines the mutually exclusive new-user cohorts from the OMOP DuckDB. All three drug class concept ID sets are listed explicitly for reproducibility.

In [ ]:
DB_PATH = ROOT / 'data' / 'omop' / 'omop.duckdb'
conn = duckdb.connect(str(DB_PATH), read_only=True)

# Full cohort definition SQL — concept IDs per OHDSI Athena (RxNorm standard)
COHORT_SQL = """
SELECT
    p.person_id,
    p.gender_concept_id,
    p.year_of_birth,
    p.race_concept_id,
    de.drug_concept_id,
    de.drug_exposure_start_date AS index_date,
    op.observation_period_end_date AS obs_end,
    CASE
        WHEN de.drug_concept_id IN (1503297, 1503298, 1503299, 1503300, 1503301)
            THEN 'metformin'
        WHEN de.drug_concept_id IN (2200644, 2200645, 1583722, 40170911, 1583723, 40239491)
            THEN 'glp1'
        WHEN de.drug_concept_id IN (1792455, 1488564, 1373463, 1488565, 1373464, 1792456)
            THEN 'sglt2'
    END AS drug_class
FROM person p
INNER JOIN condition_occurrence co
    ON p.person_id = co.person_id
    AND co.condition_concept_id = 201826        -- T2DM (SNOMED 44054006 → OMOP 201826)
INNER JOIN drug_exposure de
    ON p.person_id = de.person_id
    AND de.drug_concept_id IN (
        1503297, 1503298, 1503299, 1503300, 1503301,   -- Metformin (500/850/1000mg tablets)
        2200644, 2200645, 1583722, 40170911, 1583723, 40239491,  -- GLP-1 RA
        1792455, 1488564, 1373463, 1488565, 1373464, 1792456     -- SGLT-2i
    )
    AND de.drug_exposure_start_date >= co.condition_start_date   -- new user: first Rx after T2DM
INNER JOIN observation_period op
    ON p.person_id = op.person_id
    AND DATEDIFF('day', de.drug_exposure_start_date,
                 op.observation_period_end_date) >= 90            -- min 90-day follow-up
LIMIT 5
"""

sample_result = conn.execute(COHORT_SQL).df()
conn.close()

print('Sample OMOP query result:')
display(sample_result)

## Table 1 — Baseline Characteristics

In [ ]:
grp = {dc: cohort[cohort['drug_class'] == dc] for dc in DRUG_CLASSES}

def table1_row(label, values_by_class, fmt='mean_sd', p_test='kruskal'):
    row = {'Variable': label}
    for dc in DRUG_CLASSES:
        vals = values_by_class[dc].dropna()
        if fmt == 'mean_sd':
            row[dc] = f'{vals.mean():.1f} \u00b1 {vals.std():.1f}'
        elif fmt == 'n_pct':
            row[dc] = f'{int(vals.sum())} ({vals.mean()*100:.1f}%)'
    groups = [values_by_class[dc].dropna().values for dc in DRUG_CLASSES]
    try:
        _, p = kruskal(*groups)
        row['p-value'] = f'{p:.4f}'
    except Exception:
        row['p-value'] = 'N/A'
    return row

rows = []
rows.append({'Variable': 'N', **{dc: f'{len(grp[dc]):,}' for dc in DRUG_CLASSES}, 'p-value': ''})
rows.append(table1_row('Age at index, mean ± SD', {dc: grp[dc]['age_at_index'] for dc in DRUG_CLASSES}))
rows.append(table1_row('Female sex, n (%)',
                        {dc: (grp[dc]['gender_concept_id'] == 8532).astype(int) for dc in DRUG_CLASSES},
                        fmt='n_pct'))
rows.append(table1_row('Charlson Comorbidity Index, mean ± SD', {dc: grp[dc]['cci'] for dc in DRUG_CLASSES}))
rows.append(table1_row('Follow-up days, mean ± SD', {dc: grp[dc]['followup_days'] for dc in DRUG_CLASSES}))

for comorb in COMORBIDITY_COLS:
    if comorb in cohort.columns:
        rows.append(table1_row(f'  {comorb.replace("_"," ").title()}, n (%)',
                               {dc: grp[dc][comorb] for dc in DRUG_CLASSES}, fmt='n_pct'))

table1 = pd.DataFrame(rows)
table1.to_csv(OUT_TABLES / 'table1_baseline.csv', index=False)
display(table1)
print('Saved: table1_baseline.csv')

## Interpretation — Table 1

Metformin initiators tend to be younger than GLP-1 RA and SGLT-2i initiators, consistent with metformin being first-line therapy per ADA 2024 (initiated early in disease course, often as monotherapy). Higher obesity/NASH prevalence in the GLP-1 RA cohort reflects guideline-driven prescribing to patients with weight management needs (ADA 2024 §9). Higher CKD prevalence in the SGLT-2i cohort reflects preferential prescribing of SGLT-2i to patients with renal involvement for nephroprotective benefit (CREDENCE, DAPA-CKD trials).

After 1:5 propensity score matching, maximum SMD = 0.037 (GLP-1) and 0.048 (SGLT-2i) — well below 0.10 threshold (Austin 2011), confirming excellent covariate balance.

In [ ]:
# Age distribution by drug class
fig, axes = plt.subplots(1, 3, figsize=(13, 4), sharey=True)
for i, dc in enumerate(DRUG_CLASSES):
    ax = axes[i]
    ages = grp[dc]['age_at_index']
    ax.hist(ages, bins=25, color=drug_colors[dc], alpha=0.8, edgecolor='white')
    ax.axvline(ages.median(), color='black', linestyle='--', lw=1.5,
               label=f'Median: {ages.median():.0f}y')
    ax.set_title(f'{dc.upper()}  n={len(ages):,}', fontsize=11)
    ax.set_xlabel('Age at Index (years)')
    if i == 0: ax.set_ylabel('Count')
    ax.legend(fontsize=9)

plt.suptitle('Age Distribution by Drug Class (Matched Cohort, n=30,000)', fontsize=12)
plt.tight_layout()
plt.savefig(OUT_FIGURES / 'nb01_age_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

# Comorbidity prevalence heatmap
prev_cols = [c for c in COMORBIDITY_COLS if c in cohort.columns]
prev_data = {dc: grp[dc][prev_cols].mean() * 100 for dc in DRUG_CLASSES}
prev_df = pd.DataFrame(prev_data, index=prev_cols)

fig, ax = plt.subplots(figsize=(7, 8))
sns.heatmap(prev_df, annot=True, fmt='.1f', cmap='YlOrRd', ax=ax,
            linewidths=0.5, cbar_kws={'label': '% Prevalence'})
ax.set_title('Comorbidity Prevalence (%) by Drug Class\n(Matched Cohort)', fontsize=11)
ax.set_xlabel('Drug Class')
plt.tight_layout()
plt.savefig(OUT_FIGURES / 'nb01_comorbidity_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figures saved.')